In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
class MaskedMultiHeadAttention(nn.Module):
    """Masked multi-head attention for decoder"""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q
        
        batch_size, seq_len_q, d_model = q.shape
        seq_len_k = k.shape[1]
        seq_len_v = v.shape[1]
        
        Q = self.W_q(q)
        K = self.W_k(k)
        V = self.W_v(v)
        
        # Reshape for multi-head
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len_v, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Apply causal mask if provided
        if mask is not None:
            # truncate mask if needed to match sizes
            mask = mask[:seq_len_q, :seq_len_k]
            scores = scores.masked_fill(mask, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, d_model)
        
        output = self.W_o(attention_output)
        return output

In [3]:
class FeedForward(nn.Module):
    """Feed-forward network"""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Expand: d_model -> d_ff, then contract: d_ff -> d_model
        return self.linear2(self.relu(self.linear1(x)))

In [4]:
class TransformerEncoderBlock(nn.Module):
    """Transformer encoder block"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MaskedMultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Self-attention with residual and norm
        attn_output = self.attention(x, mask=None)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual and norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [5]:
class TransformerDecoderBlock(nn.Module):
    """Single layer of the Transformer decoder"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MaskedMultiHeadAttention(d_model, num_heads)
        self.cross_attn = MaskedMultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        self_attn_output = self.self_attn(x, mask=tgt_mask) # Masked self-attention
        x = self.norm1(x + self.dropout(self_attn_output))

        cross_attn_output = self.cross_attn(x, encoder_output, encoder_output, mask=src_mask) # Non-masked cross-attention
        x = self.norm2(x + self.dropout(cross_attn_output))

        ff_output = self.ff(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])
        
    def forward(self, x, mask=None):
        token_emb = self.embedding(x) * np.sqrt(self.d_model) # [batch_size, seq_len, d_model]
        x = self.positional_encoding(token_emb)
        
        for block in self.blocks:
            x = block(x, mask=mask)
        
        return x

In [8]:
class Decoder(nn.Module):    
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask, tgt_mask):
        token_emb = self.embedding(x) * np.sqrt(self.d_model) # [batch_size, seq_len, d_model]

        # Add positional embedding & Combine
        x = self.dropout(self.position_embedding(token_emb))
        
        # Pass through decoder blocks
        for block in self.blocks:
            x = block(x, encoder_output=encoder_output, src_mask=src_mask, tgt_mask=tgt_mask)
        
        # Final layer norm and projection
        x = self.ln_f(x)
        logits = self.head(x)
        
        return logits

In [9]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, pad_token_id, dropout=0.1, lr=1e-3):
        super().__init__()
        self.encoder  = Encoder(vocab_size, d_model, num_heads, num_layers, d_ff, dropout)
        self.decoder  = Decoder(vocab_size, d_model, num_heads, num_layers, d_ff, dropout)
        self.loss_fn   = nn.CrossEntropyLoss(ignore_index=pad_token_id)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        self.pad_token_id = pad_token_id

    def create_mask(self, src, tgt):
        src_mask = (src == self.pad_token_id).unsqueeze(1).unsqueeze(2)  # [batch_size, 1, 1, seq_len]
        tgt_mask = (tgt == self.pad_token_id).unsqueeze(1).unsqueeze(2)  # [batch_size, 1, 1, seq_len]

        seq_len_tgt = tgt.size(1)
        casual_mask = torch.triu(torch.ones(seq_len_tgt, seq_len_tgt), diagonal=1).bool()
        casual_mask = casual_mask.unsqueeze(0).unsqueeze(0)           # [1, 1, seq_len, seq_len]

        tgt_mask = tgt_mask | casual_mask  # Combine padding and causal masks

        return src_mask, tgt_mask
    
    def forward(self, src, tgt):
        src_mask, tgt_mask = self.create_mask(src, tgt)
        encoder_output = self.encoder(src, mask=src_mask)         
        logits = self.decoder(tgt, encoder_output=encoder_output, src_mask=src_mask, tgt_mask=tgt_mask)
        return logits

    def backward(self, y_pred, y_true):
        loss = self.loss_fn(y_pred.reshape(-1, y_pred.size(-1)), y_true.reshape(-1))

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()     

        return loss.item()
    
    def fit(self, src, tgt, epochs=100, print_every=10):
        self.train()

        for epoch in range(epochs):
            logits = self.forward(src, tgt[:, :-1])  # Predict next token
            loss = self.backward(logits, tgt[:, 1:])  # Compare with actual next token
            
            if (epoch + 1) % print_every == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

src_texts = [
    "I love machine learning.",
    "Transformers are powerful models.",
    "I learn deep learning."
]
tgt_texts = [
    "Toi yeu may hoc.",
    "Transformers la nhung mo hinh manh me.",
    "Toi hoc hoc sau."
]

src_tokens = tokenizer(src_texts, padding=True, truncation=True, return_tensors="pt")
tgt_tokens = tokenizer(tgt_texts, padding=True, truncation=True, return_tensors="pt")

print("Source tokens:\n", src_tokens['input_ids'])
print("Source tokens shape:", src_tokens['input_ids'].shape)
print()
print("Target tokens:\n", tgt_tokens['input_ids'])
print("Target tokens shape:", tgt_tokens['input_ids'].shape)

C:\Users\abcsd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source tokens:
 tensor([[   40,  1842,  4572,  4673,    13, 50256],
        [41762,   364,   389,  3665,  4981,    13],
        [   40,  2193,  2769,  4673,    13, 50256]])
Source tokens shape: torch.Size([3, 6])

Target tokens:
 tensor([[ 2514,    72,  9838,    84,   743, 39158,    13, 50256, 50256, 50256,
         50256, 50256, 50256],
        [41762,   364,  8591,   299, 43274,  6941,   289,   259,    71,   582,
            71,   502,    13],
        [ 2514,    72, 39158, 39158,   473,    84,    13, 50256, 50256, 50256,
         50256, 50256, 50256]])
Target tokens shape: torch.Size([3, 13])


In [11]:
trans = Transformer(
    vocab_size=tokenizer.vocab_size, 
    d_model=512, 
    num_heads=8, 
    num_layers=6, 
    d_ff=2048, 
    pad_token_id=tokenizer.pad_token_id,
    dropout=0.1
)

trans.fit(src_tokens['input_ids'], tgt_tokens['input_ids'], epochs=100, print_every=10)

Epoch 10/100, Loss: 2.9841
Epoch 20/100, Loss: 0.3654
Epoch 30/100, Loss: 0.5757
Epoch 40/100, Loss: 0.1746
Epoch 50/100, Loss: 0.1860
Epoch 60/100, Loss: 0.1425
Epoch 70/100, Loss: 0.1621
Epoch 80/100, Loss: 0.2530
Epoch 90/100, Loss: 0.1318
Epoch 100/100, Loss: 0.1041
